# Capítulo 5 · Actividad: ¿este tumor es benigno?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/exercises/chapter_05/03_actividad_breast_cancer_logistica.ipynb)

Entrena una **regresión logística** que diga si un tumor de mama es **benigno o maligno**. Haz el mismo proceso que en las actividades del Titanic y de los autos: primero entiende los datos y después elige el modelo. El preprocesamiento lo decides tú.

Cada fila es una muestra. De la imagen del núcleo celular se miden 10 cosas (radio, textura, perímetro, área, suavidad, compacidad, concavidad, puntos cóncavos, simetría y dimensión fractal). Cada una aparece tres veces:

| Grupo de columnas | Qué es |
|---|---|
| `mean ...` | Media de esa medida en la muestra |
| `... error` | Error estándar |
| `worst ...` | Media de los tres valores más grandes |
| `target` | **Target**: `0` = maligno, `1` = benigno |

`target` sale al revés de lo que uno esperaría: **1 es el caso benigno**.

Como mínimo, tu notebook debe incluir:

- La exploración de antes de modelar: forma de la tabla, balance de las dos clases, distribuciones, correlaciones y qué variables separan maligno de benigno.
- Una **regresión logística**. Las columnas no están en la misma escala (el área va por cientos y la suavidad por centésimas): decide si estandarizas.
- Métricas de **clasificación** con validación cruzada y en prueba. Como mínimo: accuracy, matriz de confusión y AUC. La accuracy sola engaña: hay más benignos que malignos.
- Tus **conclusiones**: qué modelo te quedas y por qué. Decir que un tumor maligno es benigno no es el mismo error al revés.

1. **Copia esta notebook a `workspace/`** y trabaja sobre la copia.
2. Si todavía no tienes los datos: `uv run python scripts/download_data.py`


In [ ]:
from pathlib import Path

import pandas as pd

raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
df = pd.read_csv(raiz / "data" / "breast_cancer.csv")
df.head()


## Exploración inicial

Antes de modelar hay que entender los datos: cuántas filas y columnas hay, si faltan valores y qué tan balanceadas están las clases.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             make_scorer, recall_score, roc_auc_score)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
print("Forma:", df.shape)
print("Valores nulos:", df.isna().sum().sum())
print("Duplicados:", df.duplicated().sum())

### Balance de las clases

Recuerda: `0` = maligno, `1` = benigno.

In [ ]:
print(df["target"].value_counts(normalize=True).rename(index={0: "maligno", 1: "benigno"}))
sns.countplot(data=df, x="target")
plt.xticks([0, 1], ["maligno (0)", "benigno (1)"])
plt.show()

### Distribuciones

Miramos las 10 variables `mean ...`, separadas por clase. Si dos curvas casi no se traslapan, esa variable separa bien.

In [ ]:
mean_cols = [c for c in df.columns if c.startswith("mean ")]
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for col, ax in zip(mean_cols, axes.flat):
    sns.histplot(data=df, x=col, hue="target", bins=25, kde=True, ax=ax, legend=False)
    ax.set_title(col)
plt.tight_layout()
plt.show()

### Correlaciones

Vemos qué variables se relacionan más con el `target` y si hay variables casi repetidas entre sí (multicolinealidad).

In [ ]:
corr = df.corr()
corr_target = corr["target"].drop("target").sort_values()
print(corr_target.head(6))
print(corr_target.tail(3))

plt.figure(figsize=(7, 5))
corr_target.plot.barh(color="steelblue")
plt.title("Correlación de cada variable con target")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df[mean_cols + ["target"]].corr(), annot=True, fmt=".1f", cmap="coolwarm", center=0)
plt.tight_layout()
plt.show()

### ¿Qué variables separan maligno de benigno?

In [ ]:
top = corr_target.abs().sort_values(ascending=False).index[:4]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for col, ax in zip(top, axes):
    sns.boxplot(data=df, x="target", y=col, hue="target", legend=False, ax=ax)
    ax.set_xticks([0, 1], ["maligno", "benigno"])
    ax.set_title(col)
plt.tight_layout()
plt.show()

## División en entrenamiento y prueba

Usamos `stratify=y` para que train y test conserven la misma proporción de malignos y benignos. El test se toca **una sola vez**, al final.

In [ ]:
X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(X_train.shape, X_test.shape)

## Regresión logística: ¿estandarizar o no?

Las columnas están en escalas muy distintas (el área va por cientos, la suavidad por centésimas). La regresión logística se entrena con un optimizador que sufre con eso, y la regularización penaliza distinto a cada variable según su escala. Probamos las dos opciones con validación cruzada. Meter el `StandardScaler` dentro de un `Pipeline` evita la fuga de datos: en cada fold se ajusta solo con los datos de entrenamiento de ese fold.

In [ ]:
models = {
    "Sin estandarizar": Pipeline([("model", LogisticRegression(max_iter=10000, random_state=42))]),
    "Estandarizado": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=10000, random_state=42)),
    ]),
}
rows = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=["accuracy", "roc_auc"])
    rows.append({
        "modelo": name,
        "accuracy_cv": scores["test_accuracy"].mean(),
        "accuracy_std": scores["test_accuracy"].std(),
        "auc_cv": scores["test_roc_auc"].mean(),
    })
pd.DataFrame(rows).set_index("modelo").round(4)

## Ajuste del hiperparámetro `C`

`C` controla la regularización: valores pequeños = modelo más simple. Lo elegimos con `GridSearchCV` (solo con datos de train).

In [ ]:
grid = GridSearchCV(
    models["Estandarizado"],
    param_grid={"model__C": [0.001, 0.01, 0.1, 1, 10, 100]},
    scoring="roc_auc", cv=cv,
)
grid.fit(X_train, y_train)
print("Mejor C:", grid.best_params_["model__C"])
print(f"AUC en validación cruzada: {grid.best_score_:.4f}")
best_model = grid.best_estimator_

## Métricas con validación cruzada

Además de accuracy, medimos el **recall del maligno** (de los tumores malignos, ¿cuántos detectamos?). Para eso ponemos `pos_label=0`.

In [ ]:
scoring = {
    "accuracy": "accuracy",
    "recall_maligno": make_scorer(recall_score, pos_label=0),
    "roc_auc": "roc_auc",
}
scores = cross_validate(best_model, X_train, y_train, cv=cv, scoring=scoring)
pd.DataFrame({
    "media": {k: v.mean() for k, v in scores.items() if k.startswith("test_")},
    "std": {k: v.std() for k, v in scores.items() if k.startswith("test_")},
}).round(4)

## Evaluación final en prueba

In [ ]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"AUC:      {roc_auc_score(y_test, y_proba):.4f}")
print(classification_report(y_test, y_pred, target_names=["maligno", "benigno"]))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["maligno", "benigno"], cmap="Blues"
)
plt.show()

In [ ]:
# Referencia: predecir siempre la clase mayoritaria
majority = y_train.mode()[0]
print(f"Accuracy de predecir siempre {majority} (benigno): {(y_test == majority).mean():.4f}")

## Conclusiones

- **Datos:** 569 muestras, 30 variables, sin nulos ni duplicados. Hay más benignos (63 %) que malignos (37 %), así que predecir siempre "benigno" ya da 63 % de accuracy: ese es el piso que hay que superar, y por eso la accuracy sola no basta.
- **Variables:** las que más separan las clases son las de tamaño y forma del núcleo (`worst concave points`, `worst perimeter`, `mean concave points`, `worst radius`...): correlación cercana a -0.8 con `target`, es decir, núcleos más grandes e irregulares son malignos. Muchas variables están muy correlacionadas entre sí (radio, perímetro y área miden casi lo mismo).
- **Estandarizar sí importa:** sin estandarizar la accuracy en validación cruzada fue 0.945 y con `StandardScaler` subió a 0.978 (AUC de 0.989 a 0.996), y la desviación entre folds bajó.
- **Modelo elegido:** regresión logística con estandarización y `C=1` (elegido con `GridSearchCV`). En prueba: accuracy 0.983, AUC 0.995 y solo 2 errores de 114 muestras, uno de cada tipo. Validación cruzada (0.978) y prueba (0.983) coinciden, así que no hay señales de sobreajuste.
- **Qué error importa más:** decir que un tumor maligno es benigno (falso negativo) es mucho más grave que el error contrario, porque la persona no recibiría tratamiento. Aquí el recall del maligno fue 0.965 en validación cruzada y 0.98 en prueba (1 maligno de 42 se escapó). Si quisiéramos reducir ese error aún más, podríamos bajar el umbral de `predict_proba` para clasificar como maligno con menos evidencia, a costa de más falsos positivos.
- **Limitación:** son solo 114 muestras de prueba, así que un error más o menos mueve mucho las métricas. Un modelo así apoyaría a un médico, no lo sustituiría.